In [0]:
%sql
USE CATALOG spg;
CREATE SCHEMA IF NOT EXISTS bronze;

In [0]:
%sql
-- Ingestion log
CREATE TABLE IF NOT EXISTS bronze.ingestion_log (
    read_ID BIGINT GENERATED ALWAYS AS IDENTITY,
    file_name STRING,
    src_name STRING, 
    raw_path STRING,
    load_timestamp TIMESTAMP, 
    records_written BIGINT,
    process_status STRING
)
USING DELTA;

In [0]:
%sql
-- Ingestion log
CREATE TABLE IF NOT EXISTS bronze.active_sources (
    read_ID BIGINT GENERATED ALWAYS AS IDENTITY,
    src_name STRING, 
    Active_from TIMESTAMP,
    Active_to TIMESTAMP
)
USING DELTA;

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("Key", StringType(), True),
    StructField("TicketNr", IntegerType(), True),
    StructField("ZNumber", IntegerType(), True),
    StructField("ActualDate", IntegerType(), True),
    StructField("Date", IntegerType(), True),
    StructField("Time", StringType(), True),
    StructField("UserKey", StringType(), True),
    StructField("UserId", StringType(), True),
    StructField("UserName", StringType(), True),
    StructField("PcNr", IntegerType(), True),
    StructField("PcName", StringType(), True),
    StructField("CenterKey", StringType(), True),
    StructField("CenterNr", StringType(), True),
    StructField("CenterName", StringType(), True),
    StructField("CenterLeftNr", IntegerType(), True),
    StructField("CenterRightNr", IntegerType(), True),
    StructField("Internal", StringType(), True),
    StructField("AccountKey", StringType(), True),
    StructField("AccountNr", StringType(), True),
    StructField("AccountName", StringType(), True),
    StructField("TableNr", IntegerType(), True),
    StructField("TableName", StringType(), True),
    StructField("Covers", IntegerType(), True),
    StructField("TotalPrice", DoubleType(), True),
    StructField("TotalToPay", DoubleType(), True),
    StructField("CurrencySymbol", StringType(), True),
    StructField("PrepStatus", StringType(), True),
    StructField("Receipt", StringType(), True),
    StructField("RefundedBy", StringType(), True),
    StructField("RefundFor", StringType(), True),
    StructField("LicenseInfo", StructType([
        StructField("CompanyName", StringType(), True),
        StructField("Vat", StringType(), True),
        StructField("Address", StringType(), True),
        StructField("ZipCode", StringType(), True),
        StructField("City", StringType(), True),
        StructField("Country", StringType(), True)
    ]), True),
    StructField("ReceiptNr", StringType(), True),
    StructField("PcRegistrationNr", StringType(), True),
    StructField("BlackboxSerialNr", StringType(), True),
    StructField("PrintType", StringType(), True),
    StructField("DeliveryType", IntegerType(), True),
    StructField("DeliveryDT", StringType(), True),
    StructField("Orders", ArrayType(StructType([
        StructField("Key", StringType(), True),
        StructField("ActionId", IntegerType(), True),
        StructField("ActualDate", IntegerType(), True),
        StructField("Date", IntegerType(), True),
        StructField("Time", StringType(), True),
        StructField("UserKey", StringType(), True),
        StructField("UserId", StringType(), True),
        StructField("UserName", StringType(), True),
        StructField("PcNr", IntegerType(), True),
        StructField("PcName", StringType(), True),
        StructField("TableNr", IntegerType(), True),
        StructField("TicketKey", StringType(), True),
        StructField("Lines", ArrayType(StructType([
            StructField("Key", StringType(), True),
            StructField("ProductKey", StringType(), True),
            StructField("ProductNr", IntegerType(), True),
            StructField("ProductName", StringType(), True),
            StructField("ProductType", StringType(), True),
            StructField("ProductTypeTranslated", StringType(), True),
            StructField("MenuId", StringType(), True),
            StructField("GroupKey", StringType(), True),
            StructField("GroupNr", IntegerType(), True),
            StructField("GroupName", StringType(), True),
            StructField("GroupLeftNr", IntegerType(), True),
            StructField("GroupRightNr", IntegerType(), True),
            StructField("TA", BooleanType(), True),
            StructField("Qty", DoubleType(), True),
            StructField("Price", DoubleType(), True),
            StructField("PromoName", StringType(), True),
            StructField("TotalInc", DoubleType(), True),
            StructField("TotalEx", DoubleType(), True),
            StructField("TotalDisc", DoubleType(), True),
            StructField("VatNr", IntegerType(), True),
            StructField("VatPerc", DoubleType(), True),
            StructField("CourseNr", IntegerType(), True),
            StructField("CourseName", StringType(), True),
            StructField("CostPrice", DoubleType(), True),
            StructField("Memo", StringType(), True),
            StructField("Units", DoubleType(), True),
            StructField("UnitId", IntegerType(), True)
            # StructField("Addons", 
            #             ArrayType(
            #                 StructType([])), True)  # Addons är tom array i exemplet
        ])), True),
        StructField("Paymodes", ArrayType(StructType([
            StructField("Key", StringType(), True),
            StructField("PaymodeKey", StringType(), True),
            StructField("PaymodeNr", IntegerType(), True),
            StructField("PaymodeName", StringType(), True),
            StructField("PaymodeType", StringType(), True),
            StructField("TransactionId", StringType(), True),
            StructField("TerminalId", StringType(), True),
            StructField("Memo", StringType(), True),
            StructField("GroupKey", StringType(), True),
            StructField("GroupNr", IntegerType(), True),
            StructField("GroupName", StringType(), True),
            StructField("GroupLeftNr", IntegerType(), True),
            StructField("GroupRightNr", IntegerType(), True),
            StructField("Qty", DoubleType(), True),
            StructField("Price", DoubleType(), True),
            StructField("Total", DoubleType(), True),
            StructField("Tip", DoubleType(), True),
            StructField("TransactionCost", DoubleType(), True)
        ])), True)
    ])), True)
])


In [0]:
%python
from datetime import datetime
import re
import pyspark
import os
import re
from datetime import datetime, timedelta, date
import pyspark.sql.functions as F
from pyspark.sql.functions import concat_ws, sha2
from pyspark.sql.functions import concat, col, lit, to_date

def list_all_files(base_dir):
    files = dbutils.fs.ls(base_dir)
    all_files = []
    for f in files:
        if f.isDir():
            all_files += list_all_files(f.path)
        else:
            all_files.append(f)
    return all_files

def list_files_after_date(base_dir, start_date, end_date, src):
    all_files = []
    current = start_date + timedelta(days=1)  # Start day after last read
    i = 0
    while current <= end_date:
        path = f"{base_dir}{current.year}/{str(current.month).zfill(2)}/{str(current.day).zfill(2)}/{src}"
        try:
            for f in dbutils.fs.ls(path):
                if not f.isDir():
                    all_files.append(f)
        except Exception:
            # Folder nonexisting (missing day)
            pass
        current += timedelta(days=1)
        i+=1
    return all_files

In [0]:
def logic(src, files, log_table, bronze_table, schema=None):
  src_name = src
  print(type(src))
  for file in files:
      file_path = file.path
      file_name =  file.name
      df = spark.read.schema(schema).json(file_path, multiLine=True)
      df = df.withColumn("load_time", F.current_timestamp())
      df = df.withColumn("Resturang", F.lit(src_name))

      # Kontrollera om tabellen finns
      try:
          spark.table(bronze_table)
          table_exists = True
      except AnalysisException:
          table_exists = False

      try:
        # Skriv till Delta
        if table_exists:
            df.write.format("delta") \
              .mode("append") \
              .saveAsTable(bronze_table)
            print(f"Appended {file_path} to existing table {bronze_table}")
        else:
            df.write.format("delta") \
              .mode("overwrite") \
              .option("overwriteSchema", "true") \
              .saveAsTable(bronze_table)
            print(f"Created table {bronze_table} with {file_path}")
        
        records_written = df.count()
        spark.sql(f"""
          INSERT INTO {log_table}
          (file_name, src_name, raw_path, load_timestamp, records_written, process_status)
          VALUES (
            '{file_name}',
            '{src_name}',
            '{file_path}',
            current_timestamp(),
            {records_written},
            'processed'
          )
        """)
      
      except:
        spark.sql(f"""
        INSERT INTO {log_table}
        (file_name, src_name, raw_path, load_timestamp, records_written, process_status)
        VALUES (
            '{file_name}',
            '{src_name}',
            '{file_path}',
            current_timestamp(),
            0,
            'error: {{str(e)}}'
        )
    """)

In [0]:
from pyspark.sql.utils import AnalysisException
from datetime import datetime, timedelta, date
import pyspark.sql.functions as F

catalog = 'spg'
source_path = "abfss://container@trivecstorage.dfs.core.windows.net/raw/"
log_table = f"{catalog}.bronze.ingestion_log"
active_sources_log = f"{catalog}.bronze.active_sources"

# - - - - Collect all active sources - - - - -
query = f"""
    SELECT distinct(src_name)
    FROM {active_sources_log}
"""
result_query = spark.sql(query)
active_sources = [row.src_name for row in result_query.collect()]
# - - - - - - - - - - - - - - - - - - - - - - 

for src in active_sources:
    print(src)
    bronze_table_path = f"{catalog}.bronze.trivec_combined"
    bronze_table = f"{bronze_table_path}"
    query = f"""
            SELECT MAX(CAST(load_timestamp AS timestamp)) AS last_load
            FROM {log_table}
            WHERE src_name = '{src}'
        """
    result_df = spark.sql(query)
    last_read = result_df.collect()[0]["last_load"]
    print(f"Källa: {src} Senast laddad: {last_read}")
    if last_read is None:
        last_read = datetime(2026, 1, 1)
    
    delta_time = datetime.now() - last_read

    if delta_time >= timedelta(minutes = 1):
        if delta_time < timedelta(days=1):
            print("12<  <24")
            files = list_files_after_date(source_path, last_read.date()-timedelta(days=1), datetime.now().date(), src)
        else:
            print("24 < ")
            files = list_files_after_date(source_path, last_read.date(), datetime.now().date(), src)
    else:
        files = []

    if len(files) > 0:
        logic(src, files, log_table, bronze_table, schema)
